In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tvDatafeed import TvDatafeed, Interval



In [22]:


# Login no TradingView
tv = TvDatafeed()

ticker = 'SPX'
exchange = 'TVC'


df = tv.get_hist(
    symbol=ticker,
    exchange=exchange,
    interval=Interval.in_daily,
    n_bars=10000
)
df = df[df.index.year >=2000].dropna()
df.index = pd.to_datetime(df.index).normalize().date
df.drop(columns='symbol',inplace=True)
df.index = pd.to_datetime(df.index).normalize()
df.dropna(inplace=True)
df['ret'] = df['close'].pct_change()
df.tail()




,open,high,low,close,volume,ret
2025-12-15,6860.19,6861.59,6801.49,6816.51,0.0,-0.001597
2025-12-16,6800.12,6819.27,6759.74,6800.26,0.0,-0.002384
2025-12-17,6802.88,6812.26,6720.43,6721.43,0.0,-0.011592
2025-12-18,6778.06,6816.13,6758.50,6774.76,0.0,0.007934
2025-12-19,6792.62,6840.02,6792.62,6834.50,0.0,0.008818


In [30]:
up_trigger = 1 / 100   # -1%
n = 1

df = df.dropna(subset=["ret"])
df["entry_signal"] = df["ret"] > up_trigger

position = 1
days_left = 0
positions = []

for entry in df["entry_signal"]:
    if position == 1 and entry:
        position = -1
        days_left = n
    elif position == -1:
        days_left -= 1
        if days_left == 0:
            position = 1
    positions.append(position)

df["position"] = positions

df["strategy_ret"] = df["position"].shift(1) * df["ret"]

df["strategy"] = df["strategy_ret"].cumsum()
df["buy_hold"] = df["ret"].cumsum()


In [31]:
# =========================
# PLOT
# =========================
fig = make_subplots(
    rows=1,
    cols=1,
    shared_xaxes=True,
    specs=[[{"secondary_y": True}]]
)

# Buy & Hold
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["buy_hold"] * 100,
        name="Buy & Hold",
        line=dict(width=2)
    ),
    secondary_y=False
)

# Strategy
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["strategy"] * 100,
        name="Strategy",
        line=dict(width=2)
    ),
    secondary_y=False
)



# Layout
fig.update_layout(
    title = (
        f"{ticker} | compra após queda de {up_trigger:.2%} "
        ),
    xaxis_title="Date",
    yaxis_title="Cumulative Return (%)",
    yaxis2_title="Position",
    legend=dict(x=0.01, y=0.99),
    template="plotly_white",
    height=600
)

# Ajuste do eixo secundário
fig.update_yaxes(range=[-0.05, 1.05], secondary_y=True)

fig.show()



In [32]:
total_return_bh = df["buy_hold"].iloc[-1]
total_return_strategy = df["strategy"].iloc[-1]

vol_strategy = df["strategy_ret"].std() * np.sqrt(252)
vol_bh = df["ret"].std() * np.sqrt(252)

sharpe_strategy = (
    df["strategy_ret"].mean() / df["strategy_ret"].std()
) * np.sqrt(252)

sharpe_bh = (
    df["ret"].mean() / df["ret"].std()
) * np.sqrt(252)

print("=== RESULTADOS ===")
print(f"Buy & Hold Return: {total_return_bh:.2%}")
print(f"Strategy Return:   {total_return_strategy:.2%}")
print()
print(f"Buy & Hold Vol: {vol_bh:.2%}")
print(f"Strategy Vol:   {vol_strategy:.2%}")
print()
print(f"Buy & Hold Sharpe: {sharpe_bh:.2f}")
print(f"Strategy Sharpe:   {sharpe_strategy:.2f}")


=== RESULTADOS ===
Buy & Hold Return: 203.42%
Strategy Return:   336.17%

Buy & Hold Vol: 19.37%
Strategy Vol:   19.35%

Buy & Hold Sharpe: 0.41
Strategy Sharpe:   0.67
